# ASR Shootout — Exploratory Analysis
Deep dive into benchmark results.

In [ ]:
import os, sys, json, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from src.metrics import compute_all_metrics, aggregate_results

# Load latest results
results_files = sorted(glob.glob('results/raw_results_*.json'))
if not results_files:
    print('No results found. Run: python scripts/dry_run.py')
else:
    with open(results_files[-1]) as f:
        all_results = json.load(f)
    print(f'Loaded: {results_files[-1]}')
    print(f'Models: {list(all_results.keys())}')

In [ ]:
# Flatten to DataFrame
rows = []
for model_id, results in all_results.items():
    rows.extend(results)

df = pd.DataFrame(rows)
print(df.shape)
df.head()

In [ ]:
# Summary stats per model
summary = df.groupby('model').agg(
    mean_wer=('wer', 'mean'),
    mean_cer=('cer', 'mean'),
    entity_exact_pct=('entity_exact', 'mean'),
    entity_fuzzy_pct=('entity_fuzzy', 'mean'),
    mean_entity_sim=('entity_similarity', 'mean'),
    mean_latency=('latency_ms', 'mean'),
    n=('wer', 'count'),
).round(3)

print(summary.to_string())

In [ ]:
# Entity accuracy by condition
cond_acc = df.pivot_table(
    values='entity_exact', index='condition', columns='model', aggfunc='mean'
).round(2)

plt.figure(figsize=(10, 5))
sns.heatmap(cond_acc, annot=True, fmt='.0%', cmap='RdYlGn', vmin=0, vmax=1)
plt.title('Entity Accuracy by Condition × Model')
plt.tight_layout()
plt.show()

In [ ]:
# Hardest localities
hard = df.groupby('locality')['entity_similarity'].mean().sort_values()
print('Hardest localities (avg entity_similarity across all models):')
print(hard.head(10).to_string())

In [ ]:
# Failure examples
failures = df[(df['entity_exact'] == False) & (df['entity_similarity'] < 0.6)]
print(f'\nTotal failures (sim < 0.6): {len(failures)}')
for _, row in failures.head(10).iterrows():
    print(f"  [{row['model']}] {row['locality']:<20}")
    print(f"    REF: {row['reference']}")
    print(f"    HYP: {row['hypothesis']}")
    print(f"    EarSim: {row['entity_similarity']:.2f}\n")

In [ ]:
# WER vs Latency tradeoff
fig, ax = plt.subplots(figsize=(8, 5))
for model_id in df['model'].unique():
    sub = df[df['model'] == model_id]
    ax.scatter(sub['latency_ms'].mean(), sub['wer'].mean(), s=200, label=model_id)
ax.set_xlabel('Avg Latency (ms)')
ax.set_ylabel('Avg WER')
ax.set_title('Accuracy vs Speed Tradeoff')
ax.legend()
plt.tight_layout()
plt.show()